# Lesson 3.6 — Connection Protocols and Progressive Discovery

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Compare MCP, direct API/CLI, and agent-to-agent integration mechanisms
- Select a protocol based on reuse, governance, and coupling needs
- Weigh progressive discovery against monolithic-context strategies

**You will produce:** A connection-protocol decision matrix and a progressive-discovery plan.

## Concepts

How Claude connects to tools and data is an architectural choice:

- **MCP (Model Context Protocol)** — an open standard for exposing tools/data/resources to models through a uniform server interface. Best when you want reusable, governable, swappable integrations shared across agents and clients.
- **Direct API/CLI** — call a service's REST API or a CLI directly from your code. Simplest for a one-off, tightly-coupled integration you fully control.
- **Agent-to-agent (A2A)** — one agent delegates to another autonomous agent. Powerful for cross-team/cross-org capabilities but adds trust, latency, and coordination complexity.

**Progressive discovery vs monolithic context:** rather than stuffing *all* tools/docs into context up front (monolithic — expensive, noisy, and bloats the window), expose capabilities so the model **discovers and loads them as needed** (progressive). Progressive discovery keeps context lean, improves tool selection, and scales to large tool/data catalogs.

> **Reference:** [Anthropic — Model Context Protocol (MCP)](https://docs.anthropic.com/en/docs/build-with-claude/mcp)

In [ ]:
# !pip install anthropic
import os, time, json

try:
    import anthropic
    if os.environ.get("ANTHROPIC_API_KEY"):
        client = anthropic.Anthropic(); LIVE = True
    else:
        client = None; LIVE = False
except Exception as e:
    client = None; LIVE = False
    print("anthropic SDK unavailable:", e)

MODELS = {"opus":"claude-opus-4-20250514","sonnet":"claude-sonnet-4-20250514","haiku":"claude-3-5-haiku-20241022"}

def ask_claude(prompt, model=MODELS["sonnet"], system=None, max_tokens=512):
    if not LIVE:
        return f"[MOCK {model}] would answer: {prompt[:80]}..."
    msg = client.messages.create(model=model, max_tokens=max_tokens,
        system=system or "You are a helpful assistant.",
        messages=[{"role":"user","content":prompt}])
    return msg.content[0].text

print("LIVE" if LIVE else "MOCK", "mode ready")

## Code-Along: Step 1 — Build the protocol decision matrix

We score each mechanism on reuse, governance, coupling, and complexity, then select by the dominant requirement.

In [ ]:
def choose_protocol(reuse_across_clients, strong_governance, one_off, autonomous_partner):
    if autonomous_partner:
        return "agent-to-agent", "Delegating to another autonomous agent/team"
    if reuse_across_clients or strong_governance:
        return "MCP", "Reusable, governable, swappable tool/data interface"
    if one_off:
        return "direct API/CLI", "Simple, tightly-coupled, fully controlled integration"
    return "direct API/CLI", "Default to simplest until reuse/governance demands MCP"

print(choose_protocol(True, True, False, False))
print(choose_protocol(False, False, True, False))
print(choose_protocol(False, False, False, True))

## Code-Along: Step 2 — Plan progressive discovery over a tool catalog

With many tools, load them on demand instead of all at once. We simulate selecting a small relevant subset per query to keep the context window lean.

In [ ]:
catalog = ["read_ticket","policy_search","member_lookup","refund_request",
           "claims_history","drug_formulary","appointment_book","translate"]

def progressive_select(query, catalog, k=3):
    # Toy relevance: keyword overlap; production uses embeddings/tool descriptions.
    scored = sorted(catalog, key=lambda t: sum(w in t for w in query.lower().split()), reverse=True)
    return scored[:k]

print("monolithic context would load all", len(catalog), "tools every call")
print("progressive loads only:", progressive_select("look up member policy", catalog))

## Exercise

Extend `choose_protocol` with a **latency-critical** flag that penalizes agent-to-agent (extra network hops). Then write 3 scenarios (internal reusable tools; a quick script against one SaaS API; delegating to a partner's fraud-scoring agent) and defend each pick in one line — this is exactly the Domain 3 'evaluate connection protocols' objective.

In [ ]:
# TODO: add latency_critical to choose_protocol and adjust A2A selection
# TODO: run the 3 scenarios and print protocol + one-line justification each

## Key Takeaways

- MCP for reusable/governable integrations; API/CLI for simple one-offs; A2A for autonomous delegation.
- Default to the simplest mechanism until reuse or governance justifies MCP.
- Agent-to-agent adds trust, latency, and coordination cost — use deliberately.
- Progressive discovery keeps context lean and tool selection accurate vs monolithic context.

---

**Phase complete!** Return to the [study plan README](../README.md) to continue.